In [1]:
import sys

# 1. Gỡ sạch các thư viện đang bị hỏng trên chính Kernel này
!{sys.executable} -m pip uninstall -y transformers sentence-transformers huggingface_hub

# 2. Xóa bộ nhớ đệm (cache) của pip để nó không lôi lại cái file lỗi cũ ra cài
!{sys.executable} -m pip cache purge

# 3. Cài lại bản ổn định nhất (khóa luôn bản transformers 4.38.2 cực kỳ ổn định)
!{sys.executable} -m pip install transformers==4.38.2 sentence-transformers torch tf-keras tensorflow langdetect emoji wordcloud

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 5.4.1
Uninstalling sentence-transformers-5.4.1:
  Successfully uninstalled sentence-transformers-5.4.1
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Files removed: 2522 (7969.1 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of sentence-transformers to determine which version is compatible w

  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [63 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp313-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\Lenovo\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\Lenovo\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\Lenovo\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to minimal
      info: setting default host tripl

In [2]:
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

# --- 1. Thiết lập môi trường (Chạy trước khi gọi TensorFlow/Keras) ---
os.environ["TF_USE_LEGACY_KERAS"] = "1" # Ép dùng Keras 2 cho HuggingFace BERT

# --- 2. Core & Utilities ---
import string
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from collections import Counter

# --- 3. Visualization ---
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import seaborn as sns
from wordcloud import WordCloud

# --- 4. NLP & Text Processing ---
import re
import emoji
from langdetect import detect, LangDetectException, DetectorFactory
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

# Khởi tạo (nếu bạn cần gọi trực tiếp trong notebook)
DetectorFactory.seed = 0

# --- 5. Machine Learning (Scikit-Learn & SciPy) ---
from scipy.stats import spearmanr
from scipy.sparse import hstack, csr_matrix

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# --- 6. Deep Learning & Transformer Models ---
import tensorflow as tf
import tf_keras
from tf_keras.models import Sequential
from tf_keras.layers import Input, LSTM, SimpleRNN, Dense, Dropout, Embedding, TextVectorization
from tf_keras.callbacks import EarlyStopping

from sentence_transformers import SentenceTransformer
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer

# --- 7. Import các hàm từ Modules Local (src) ---
from src.cleaning import (
    clean_and_convert_labels, 
    identify_language, 
    perform_train_test_split
)

from src.preprocessing import (
    get_wordnet_pos, 
    emoji_cleaning, 
    delete_repeated_char, 
    preprocess_and_tokenize
)

from src.models import (
    compute_comprehensive_metrics, 
    balance_data, 
    evaluate_model,
    CombinedExtractor, 
    SklearnPipeline, 
    LSTMPipeline, 
    TFBERTPipeline
)

from src.eda import (
    plot_label_distribution, 
    create_text_features, 
    plot_overall_text_features_distribution,
    plot_feature_distributions_by_label, 
    print_detailed_statistics_by_label,
    plot_feature_trend_by_label, 
    plot_emoji_analysis, 
    plot_overall_top_words,
    get_ngram_description, 
    get_ngram_filename_suffix, 
    plot_top_words_by_label,
    plot_tfidf_features_per_class, 
    get_vocab_set, 
    jaccard, 
    plot_jaccard_similarity_heatmap
)

ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:
import tensorflow as tf
import torch

# 1. Cấu hình GPU cho TensorFlow (Dùng cho LSTM và TFBERTPipeline)
tf_gpus = tf.config.list_physical_devices('GPU')
if tf_gpus:
    print(f"TENSORFLOW: Đã tìm thấy GPU.")
    # Bật tính năng Memory Growth: Chỉ cấp phát VRAM khi cần thiết
    try:
        for gpu in tf_gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Đã kích hoạt Memory Growth cho TensorFlow.")
    except RuntimeError as e:
        print("Lỗi khi cấu hình Memory Growth:", e)
else:
    print("TENSORFLOW: Không có GPU. Đang chạy bằng CPU.")

# 2. Cấu hình GPU cho PyTorch (Dùng cho SentenceTransformer/BERT Vector)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"PYTORCH: Đã tìm thấy GPU ({gpu_name}).")
else:
    print("PYTORCH: Không có GPU. Đang chạy bằng CPU.")

In [ ]:
import kagglehub
shymammoth_shopee_reviews_path = kagglehub.dataset_download('shymammoth/shopee-reviews')

print('Data source import complete.')


# Loading data

In [ ]:
DATA_PATH = shymammoth_shopee_reviews_path + "/shopee_reviews.csv"

print("Loading dataset...")
df = pd.read_csv(DATA_PATH)

# Randomly sample 100,000 rows
df = df.sample(frac=100_000/len(df), random_state=42).reset_index(drop=True)

print(f"Sampled dataset shape: {df.shape}")

In [ ]:
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'

# Cleaning data

## Resolving Data Type Heterogeneity in Target Labels ##

In [ ]:
print("="*45)
print("-------- Unique labels --------")
print(df[LABEL_COLUMN].value_counts())
print("="*45)
print("-------- Checking types --------")
print(df[LABEL_COLUMN].apply(type).value_counts())
print("="*45)

In [ ]:
df = clean_and_convert_labels(df)

In [ ]:
DetectorFactory.seed = 42
tqdm.pandas(desc="Identifying Languages")

print("--- [STATUS] Initializing Full-Scale Corpus Analysis ---")
original_row_count = len(df)

# 1. Processing: Language identification on the entire Dataset
# We skip the sampling step to ensure the integrity of the input data
df['lang'] = df['text'].fillna('').astype(str).progress_apply(identify_language)

# 2. Statistical Distribution: Calculate distribution ratio
lang_stats = df['lang'].value_counts(normalize=True).mul(100).round(2)
en_ratio = lang_stats.get('en', 0)
others_ratio = 100 - en_ratio

fig, ax = plt.subplots(figsize=(10, 7), dpi=100)

# Draw Donut chart
wedges, _ = ax.pie(
    [en_ratio, others_ratio],
    colors=['blue', '#D5D8DC'], # Changed color from colors[0] to 'blue'
    startangle=140,
    wedgeprops={'width': 0.45, 'edgecolor': 'w', 'linewidth': 0},
    explode=(0.05, 0)
)

# Add KPI to the center of the circle (Central Metric)
ax.text(0, 0.05, f"{en_ratio:.1f}%", ha='center', va='center',
        fontsize=38, fontweight='bold', color='blue') # Changed color from colors[0] to 'blue'
ax.text(0, -0.15, "English Density", ha='center', va='center',
        fontsize=12, color='#2C3E50', fontweight='semibold', alpha=0.7)

# Title and In-depth Description (Description)
plt.title('Corpus Linguistic Composition', fontsize=18, pad=25, fontweight='bold', color='#2C3E50')

plt.axis('equal')
plt.tight_layout()
plt.show()

# 4. Automated Filtration: Filter data based on statistical threshold
STRICT_THRESHOLD = 90.0
is_dominant = en_ratio > STRICT_THRESHOLD

if is_dominant:
    print(f"\n[ACTION] English dominance detected (> {STRICT_THRESHOLD}%). Isolating primary corpus...")
    df = df[df['lang'] == 'en'].reset_index(drop=True)
    print(f"Filtration successful. Current Dimension: {len(df)} rows.")
else:
    print(f"\n[OBSERVATION] Multilingual distribution detected. Maintaining 'lang' feature for stratified analysis.")

# Memory cleanup: Remove intermediate column after Feature Engineering is complete
if 'lang' in df.columns:
    df = df.drop('lang', axis=1)
    print("[CLEANUP] Internal feature 'lang' decommissioned.")

print("--- [STATUS] Pipeline Execution Completed ---")

In [ ]:
X_train, X_test, y_train, y_test = perform_train_test_split(df, LABEL_COLUMN, test_size=0.20, random_state=42)

# EDA

In [ ]:
df_eda = X_train.copy()
df_eda[LABEL_COLUMN] = y_train.copy()

In [ ]:
print("="*45)
print("Shape:", df.shape)

print("="*45)
print("Columns:", df.columns.tolist())

print("="*45)
print(f'Dtypes:\n{df.dtypes}')

print("="*45)
print(f'Info:')
df.info()

print("="*45)
print(f'Null values:\n{df.isnull().sum()}')

print("="*45)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate texts:", df['text'].duplicated().sum())

print("="*45)
print("Sample:")
df_eda.head(10)

In [ ]:
LABEL_ORDER = sorted(df['label'].unique())
LABEL_COUNT= len(LABEL_ORDER)

theme_palette='inferno'
colors = sns.color_palette(theme_palette, LABEL_COUNT).as_hex()

print(LABEL_ORDER)

nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))

## Target Class Frequency & Prevalence Profiling ##

In [ ]:
plot_label_distribution(df_eda, LABEL_COLUMN, LABEL_ORDER, colors, theme_palette)

## Quantitative Textual Patterns ##

In [ ]:
# Execution example (Replace 'text' with your actual column name)
df_eda=create_text_features(df_eda)

In [ ]:
df.head(10)

In [ ]:
features=['n_words', 'num_emoji', 'upper_ratio']

## Quantitative Analysis of Review Length ##

In [ ]:
# Call the new function for 'n_words' and 'n_chars'
plot_overall_text_features_distribution(df_eda, ['n_words'], theme_palette)

## Across Labels

In [ ]:
# --- Call the new combined plotting function for 'n_words' ---
plot_feature_distributions_by_label(
    df=df_eda,
    feature_col='n_words',
    label_column=LABEL_COLUMN,
    label_order=LABEL_ORDER,
    theme_palette=theme_palette
)

In [ ]:
# Call the new function with your dataframe and relevant variables
print_detailed_statistics_by_label(df_eda, LABEL_COLUMN, ['n_words'])

## Spearman correlation ##

In [ ]:
# Call the new function with your dataframe and relevant variables
plot_feature_trend_by_label(df_eda, features, LABEL_COLUMN, LABEL_ORDER, colors)

## Emoji ##

In [ ]:
plot_emoji_analysis(df_eda, TEXT_COLUMN, LABEL_COLUMN, LABEL_ORDER, colors)

## Global top words ##

In [ ]:
# --- GỌI HÀM THỰC THI (EDA GLOBAL BAG OF WORDS) ---

print("="*50)
print("EDA: GLOBAL BAG OF WORDS FREQUENCY")
print("="*50)

# Chạy cho Unigram
plot_overall_top_words(df_eda, 'processed_text', top_n=20, ngram_range=(1, 1), title="Unigrams")

# Chạy cho Bigram
plot_overall_top_words(df_eda, 'processed_text', top_n=20, ngram_range=(2, 2), title="Bigrams")

# Chạy cho Trigram
plot_overall_top_words(df_eda, 'processed_text', top_n=20, ngram_range=(3, 3), title="Trigrams")

## Class-Specific Top Words

In [ ]:
# Unigram
plot_top_words_by_label(df_eda, 'processed_text', LABEL_COLUMN, top_n=15, ngram_range=(1,1))

# Bigram
plot_top_words_by_label(df_eda, 'processed_text', LABEL_COLUMN, top_n=15, ngram_range=(2,2))

# Trigram
plot_top_words_by_label(df_eda, 'processed_text', LABEL_COLUMN, top_n=15, ngram_range=(3,3))

# Unigram, Bigram, and Trigram combined
plot_top_words_by_label(df_eda, 'processed_text', LABEL_COLUMN, top_n=15, ngram_range=(1,3))

In [ ]:
# Call the new function with your dataframe and relevant variables
plot_tfidf_features_per_class(df_eda, 'processed_text', LABEL_COLUMN, LABEL_ORDER, colors, theme_palette)

## Ordinal analysis

In [ ]:
# Call the new function with your dataframe and relevant variables
plot_jaccard_similarity_heatmap(df_eda, LABEL_COLUMN, LABEL_ORDER, theme_palette)

# Preprocessing

In [ ]:
X_train_features = create_text_features(X_train, text_col='text')
X_test_features = create_text_features(X_test, text_col='text')

In [ ]:
# Initialize Lemmatizer (Should be outside the function to avoid reloading multiple times, which speeds up apply operations on DataFrame)
lemmatizer = WordNetLemmatizer()

# Download necessary NLTK data packages
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng') 
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

X_train_features['processed_text'] = X_train_features[TEXT_COLUMN].apply(preprocess_and_tokenize)
X_test_features['processed_text'] = X_test_features[TEXT_COLUMN].apply(preprocess_and_tokenize)

In [ ]:
df_eda = X_train_features.copy()
df_eda['label'] = y_train

# Data Splitting & Pipeline Architecture

In [ ]:
df_train = X_train_features.copy()
df_train['label'] = y_train.values # Gộp y vào để balance_data có thể hoạt động

df_test = X_test_features.copy()
df_test['label'] = y_test.values   # Khởi tạo df_test cho Lỗi 4

# 2. Tạo tập validation từ tập train (cho Lỗi 1)
from sklearn.model_selection import train_test_split

df_train, df_val, y_train, y_val = train_test_split(
    df_train,
    df_train['label'],
    test_size=0.1,
    random_state=42,
    stratify=df_train['label']
)

In [ ]:
extractors = {
    "BoW (1-3 gram)": {
        "ext": CountVectorizer(ngram_range=(1, 3), max_features=10000),
        "scaler": MaxAbsScaler()
    },
    "TF-IDF (1-3 gram)": {
        "ext": TfidfVectorizer(ngram_range=(1, 3), max_features=10000),
        "scaler": MaxAbsScaler()
    },
    "BERT Vector": {
        "ext": SentenceTransformer('all-MiniLM-L6-v2'),
        "scaler": MinMaxScaler()
    },
    "Hybrid (TF-IDF + BERT)": {
        "ext": CombinedExtractor(max_features=5000),
        "scaler": MaxAbsScaler()
    }
}

models = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "NaiveBayes": MultinomialNB()
}

experiments = {}
exp_counter = 1

# Sinh tổ hợp
for ext_name, config in extractors.items():
    for mod_name, model in models.items():
        if ext_name == "Hybrid (TF-IDF + BERT)" and mod_name == "NaiveBayes":
            continue

        experiments[f"{exp_counter}. {ext_name} + {mod_name} (Class Weights)"] = SklearnPipeline(
            extractor=config["ext"],
            model=model,
            text_col='text' if "BERT" in ext_name else 'processed_text',
            scaler=config["scaler"]
        )
        exp_counter += 1

# Deep Learning Pipelines
experiments[f"{exp_counter}. LSTM Deep Learning (Class Weights)"] = LSTMPipeline(
    max_vocab=15000, max_len=100, text_col='text' # Đổi sang văn bản thô cho LSTM
)
exp_counter += 1

experiments[f"{exp_counter}. DistilBERT Fine-Tuning (Class Weights)"] = TFBERTPipeline(
    model_name='distilbert-base-uncased', max_len=128, text_col='text'
)

In [ ]:
import os
import time
import pandas as pd

my_class_names = sorted(np.unique(y_test))

# --- 1. SETUP FILE LƯU TIẾN TRÌNH ---
save_file = "training_checkpoint_results.csv"

if os.path.exists(save_file):
    # Nếu file đã tồn tại, đọc kết quả cũ lên
    df_results = pd.read_csv(save_file)
    results = df_results.to_dict('records')
    completed_experiments = df_results["Tổ hợp Model"].tolist()
    print(f"Tìm thấy dữ liệu cũ! Đã có {len(completed_experiments)} tổ hợp hoàn thành.")
else:
    # Nếu chạy lần đầu tiên
    results = []
    completed_experiments = []
    print("Bắt đầu chạy mới hoàn toàn.")

print(f"Tổng số tổ hợp cần huấn luyện: {len(experiments)}\n" + "="*60)

# --- 2. VÒNG LẶP HUẤN LUYỆN ---
for exp_name, pipeline in experiments.items():
    # Kiểm tra xem mô hình này đã chạy chưa
    if exp_name in completed_experiments:
        print(f"\nBỎ QUA: [ {exp_name} ] (Đã huấn luyện trước đó)")
        continue

    print(f"\n ĐANG HUẤN LUYỆN: [ {exp_name} ]...")

    # Huấn luyện
    start_time = time.time()
    pipeline.fit(df_train, y_train, df_val, y_val)
    train_time = time.time() - start_time

    # Suy luận
    start_time = time.time()
    y_pred = pipeline.predict(df_test)
    inf_time = time.time() - start_time

    # Gọi hàm vẽ Confusion Matrix và in Report
    evaluate_model(y_test, y_pred, exp_name, class_names=my_class_names)

    # Tính toán Metrics tổng hợp
    metrics = compute_comprehensive_metrics(y_test, y_pred)

    # Thêm vào danh sách kết quả
    new_result = {
        "Tổ hợp Model": exp_name,
        "Thời gian Train (s)": round(train_time, 2),
        "Thời gian Test (s)": round(inf_time, 2),
        **metrics
    }
    results.append(new_result)

    # --- SỬA Ở ĐÂY: LƯU XUỐNG Ổ CỨNG NGAY LẬP TỨC ---
    df_temp = pd.DataFrame(results)
    df_temp.to_csv(save_file, index=False)
    print(f" Đã lưu tiến trình cho: {exp_name} vào file CSV.")

# --- 3. IN BẢNG TỔNG HỢP CUỐI CÙNG ---
print("\n ĐÃ HOÀN THÀNH TOÀN BỘ TỔ HỢP!")
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="F1_Macro", ascending=False).reset_index(drop=True)

print("\n BẢNG TỔNG HỢP KẾT QUẢ CỦA TẤT CẢ CÁC TỔ HỢP:\n")
display(df_results)